In [0]:
df = spark.read.table("ml_catalog.gold.churn")
display(df)


In [0]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [col for col in df.columns if col != "Exited"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df = assembler.transform(df)
df.select("features", "Exited").display()

In [0]:
!pip install xgboost
from xgboost.spark import SparkXGBClassifier


In [0]:
train, test = df.randomSplit([0.8, 0.2], seed=42)

In [0]:
train.display()

In [0]:
test.display()

In [0]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="Exited"
)

model = gbt.fit(train)
predictions = model.transform(test)

predictions.select("Exited", "prediction").display()

In [0]:
type(predictions)

In [0]:
cm = predictions.groupBy("Exited") \
    .pivot("prediction") \
    .count() \
    .fillna(0)

cm.display()

In [0]:
rows = cm.collect()

In [0]:
type(rows)


In [0]:
cm.columns

In [0]:
TN = cm_dict[0]["0.0"]
FP = cm_dict[0]["1.0"]
FN = cm_dict[1]["0.0"]
TP = cm_dict[1]["1.0"]

In [0]:
precision = TP / (TP + FP)
recall = TP / (TP + FN)
accuracy = (TP + TN) / (TP + TN + FP + FN)

precision, recall, accuracy

In [0]:
%sql SHOW VOLUMES IN ml_catalog.ml;

In [0]:
spark.sql("USE CATALOG ml_catalog")
spark.sql("USE SCHEMA ml")

In [0]:
import os
import mlflow
import mlflow.spark
from mlflow.models.signature import infer_signature
from pyspark.ml.functions import vector_to_array

# ✅ IMPORTANT FIX
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/ml_catalog/ml/volume_models/"

# convert features
train_fixed = train.withColumn("features", vector_to_array("features"))

# input example
input_example = train_fixed.limit(5).toPandas().drop("Exited", axis=1)

# prediction example
pred_example = model.transform(train).select("prediction").limit(5).toPandas()

# signature
signature = infer_signature(input_example, pred_example)

with mlflow.start_run():

    mlflow.spark.log_model(
        spark_model=model,
        artifact_path="churn-modelsignature",
        signature=signature,
        input_example=input_example
    )